# Average feature importance across all urban forms for every BSU

In [ ]:
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib_scalebar.scalebar import ScaleBar

## Load models

In [ ]:
fi = {}
lc = {}
perf = []

for reduction in ["fa", "pca"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = model.feature_importances_
                else:
                    lc[reduction][model_type][cluster] = model.local_coef_
                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,

                        }
                    )
                )

## Load Data

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Specialisté",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Počet obyvatel na dům",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s dlouhodobým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]
fas = census[selection]

## Merge data with clusters

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
lc_mean = pd.concat(lc["fa"]["lr"]).groupby(level=1).mean()
lc_std = pd.concat(lc["fa"]["lr"]).groupby(level=1).std()

In [ ]:
lc_mean_mean = lc_mean.mean(axis=0)
lc_mean_mean.reindex(lc_mean_mean.abs().sort_values(ascending=False).index).head(5)

In [ ]:
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].abs().mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
## nejvyšší vliv mají obecně tyhle proměnný
abs_mean.mean(axis=1).sort_values(ascending=False).head(10)

In [ ]:
dfs = []

for i in list(lc["fa"]["lr"].keys()):
    imp = lc["fa"]["lr"][i].mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

mean = pd.concat(dfs, axis=1)
mean.std(axis=1).sort_values(ascending=False).head(10)

In [ ]:
lc_mean = lc_mean.set_geometry(data.geometry)
lc_std = lc_std.set_geometry(data.geometry)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))


lc_mean.plot(
    ax=axes[0],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="coolwarm_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[0].set_axis_off()
axes[0].set_title("mean")


lc_std.plot(
    ax=axes[1],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    legend=True,
    cmap="YlOrRd",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[1].set_axis_off()
axes[1].set_title("std")

plt.tight_layout()
# fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")
plt.show()

## Calculate the avg and std for feature importances

In [ ]:
fi_mean = pd.concat(fi["fa"]["rf"])
fi_std = pd.concat(fi["fa"]["rf"]).groupby(level=1).std()

In [ ]:
rename_dict = {
    "Počet obyvatel na dům": "Population Density",

    "Obyvatelstvo - věk: 7 - 14  - celkem": "Age Group: 7-14",
    "Obyvatelstvo - věk: 15 - 24  - celkem": "Age Group: 15-24",
    "Obyvatelstvo - věk: 45 - 54  - celkem": "Age Group: 45-54",

    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem":
        "Secondary education with graduation",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem":
        "Secondary education without graduation",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem":
        "Unknown Education",

    "Obyvatelstvo - státní občanství: Slovenská republika - celkem":
        "Citizenship - Slovakia",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem":
        "Citizenship - EU",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem":
        "Citizenship - Unknown",

    "Obyvatelstvo - s dlouhodobým pobytem - celkem":
        "Long-Term Stay Residents",

    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem":
        "Religion - Non-Religious",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem":
        "Religion - Unspecified",

    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem":
        "Marital Status - Married",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem":
        "Marital Status - Divorced",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem":
        "Marital Status - Widowed",

    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem":
        "Empl. sector - Agriculture/Fishery",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem":
        "Empl. sector - Industry",

    "Zaměstnaní - Specialisté":
        "Empl. occupation - Specialists",
    "Zaměstnaní - Pracovníci ve službách a prodeji":
        "Empl. occupation - Service/Sales",
    "Zaměstnaní - Řemeslníci a opraváři":
        "Empl. occupation - Craft/Repair",

    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem":
        "Empl. status - Employees",

    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem":
        "Econ. Activity - Parental Leave",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem":
        "Econ. Activity - Dependent Population",

    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví":
        "Residents in Owner-Occupied Dwellings",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý":
        "Residents in Rented Dwelling",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní":
        "Residents in Cooperative Dwellings",

    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba":
        "Residents in Privately Owned Houses"
}


In [ ]:
fi_mean = fi_mean.rename(columns=rename_dict)

In [ ]:
fi_mean= fi_mean.rename(
    index={
        1: "Incoherent Large-Scale Homogeneous Fabric",
        2: "Incoherent Large-Scale Heterogeneous Fabric",
        3: "Incoherent Small-Scale Linear Fabric",
        4: "Incoherent Small-Scale Sparse Fabric",
        5: "Incoherent Small-Scale Compact Fabric",
        6: "Coherent Interconnected Fabric",
        7: "Coherent Dense Disjoint Fabric",
        8: "Coherent Dense Adjacent Fabric",
    }
)

In [ ]:
fi_m = fi_mean.mean(axis=0)
fi_m

In [ ]:
var_category = {
    # Demographics
    "Age Group: 7-14": "Demographics",
    "Age Group: 15-24": "Demographics",
    "Age Group: 45-54": "Demographics",
    "Citizenship - Slovakia": "Demographics",
    "Citizenship - EU": "Demographics",
    "Citizenship - Unknown": "Demographics",
    "Long-Term Stay Residents": "Demographics",
    "Religion - Non-Religious": "Demographics",
    "Religion - Unspecified": "Demographics",
    "Marital Status - Married": "Demographics",
    "Marital Status - Divorced": "Demographics",
    "Marital Status - Widowed": "Demographics",

    # Socioeconomic
    "Secondary education with graduation": "Socioeconomic",
    "Secondary education without graduation": "Socioeconomic",
    "Unknown Education": "Socioeconomic",
    "Empl. sector - Agriculture/Fishery": "Socioeconomic",
    "Empl. sector - Industry": "Socioeconomic",
    "Empl. occupation - Specialists": "Socioeconomic",
    "Empl. occupation - Service/Sales": "Socioeconomic",
    "Empl. occupation - Craft/Repair": "Socioeconomic",
    "Empl. status - Employees": "Socioeconomic",
    "Econ. Activity - Parental Leave": "Socioeconomic",
    "Econ. Activity - Dependent Population": "Socioeconomic",

    # Housing
    "Residents in Owner-Occupied Dwellings": "Housing",
    "Residents in Rented Dwelling": "Housing",
    "Residents in Cooperative Dwellings": "Housing",
    "Residents in Privately Owned Houses": "Housing",
    "Population Density": "Housing",
}
category_palette = {
    "Demographics": "#b2cd32",     # blue
    "Socioeconomic": "#7CBAE4",    # orange
    "Housing": "#ECBF43",          # green
}
ordered_cols = fi_m.sort_values(ascending=False).index

palette = [
    category_palette[var_category[col]]
    for col in ordered_cols
]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))



plt.xticks(rotation=90)

sns.boxplot(
    data=fi_mean.loc[:, fi_m.sort_values(ascending=False).index],
    ax=ax,
    palette=palette,
    linecolor='k',
    linewidth=0.8,
    showfliers = False,
    medianprops={"color": "k", "linewidth": 1.2},
    saturation=1,
)
sns.despine()

In [ ]:
from scipy import stats

In [ ]:
jitter = 0.06
x_data = [np.array([i] * len(fi_mean)) for i, d in enumerate(fi_mean.columns)]
x_jittered = [x + stats.t(df=6, scale=jitter).rvs(len(x)) for x in x_data]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

medianprops = dict(
    linewidth=1.2,
    color="k",
    solid_capstyle="butt"
)
boxprops = dict(
    linewidth=0.8,
    color="k"
)

ax.boxplot(
    fi_mean.loc[:, fi_m.sort_values(ascending=False).index].dropna(),
    positions=range(len(fi_mean.columns)),
    showfliers = False, # Do not show the outliers beyond the caps.
    showcaps = False,   # Do not show the caps
    tick_labels=fi_mean.columns,
    medianprops = medianprops,
    whiskerprops = boxprops,
    boxprops = boxprops,
)

for x, col in enumerate(fi_m.sort_values(ascending=False).index):
     ax.scatter(x_jittered[x],fi_mean[col],c=palette[x], s = .005, alpha=0.1)


ax.tick_params(axis="x", pad=6)

for lab in ax.get_xticklabels():
    lab.set_rotation(90)
    lab.set_ha("right")
    lab.set_rotation_mode("anchor")
sns.despine()


In [ ]:
fi_mean.dropna()

In [ ]:
fi_mean_m = fi_mean.median(axis=0)
fi_mean_m.sort_values(ascending=False).head(30)

In [ ]:
fi_mean_std = fi_std.mean(axis=0)
fi_mean_std.sort_values(ascending=False).head(30)

In [ ]:
fi_std_ratio.mean(axis=0).sort_values(ascending=False).head(10)

In [ ]:
dfs = []

for i in list(fi["fa"]["rf"].keys()):
    imp = fi["fa"]["rf"][i].abs().mean(axis=0)
    imp = pd.DataFrame(imp, columns=[str(i)])
    dfs.append(imp)

abs_mean = pd.concat(dfs, axis=1)
pd.DataFrame(abs_mean.mean(axis=1).sort_values(ascending=False).head(15).round(4))

Get the variables with overall highest importance

Get the variables with overall highest std deviation of the importance

In [ ]:
dfs = []

for i in list(fi["fa"]["rf"].keys()):
    # std / mean per feature for model i
    imp_std = fi["fa"]["rf"][i].std(axis=0)
    imp_mean = fi["fa"]["rf"][i].mean(axis=0)

    imp_ratio = imp_std / imp_mean
    imp_ratio = pd.DataFrame(imp_ratio, columns=[str(i)])

    dfs.append(imp_ratio)

ratio = pd.concat(dfs, axis=1)

a =pd.DataFrame(
    ratio.mean(axis=1)
    .sort_values(ascending=False)
    .head(15)
)
a


In [ ]:
c = pd.concat([b.rename(columns={0:"a"}),a],axis=1)

In [ ]:
c["b"] = c["a"]*100
c["c"] = c[0]*10

In [ ]:
c["sum"] = c["b"]+c["c"]

In [ ]:
c["a"].quantile([0.25,0.7])

In [ ]:
c[0].quantile([0.25,0.7])

In [ ]:
d = c.loc[(c["a"]>0.0338) & (c[0]>0.398101)].head(30)
d

In [ ]:
e = c.loc[(c["a"]>0.0438) & (c[0]>0.338207)].head(30)
e

### Assign geometry

In [ ]:
fi_mean = fi_mean.set_geometry(data.geometry)
fi_std = fi_std.set_geometry(data.geometry)

In [ ]:
ax = fi_mean.plot(
    figsize=(15, 10),
    column="Počet obyvatel na dům",
    legend=True,
    cmap="RdYlGn_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    #  scheme="natural_breaks"
)
ax.set_axis_off()

fig = ax.get_figure()
# fig.savefig("rentals_imp.png", dpi=300)

In [ ]:
ax = fi_mean.plot(
    figsize=(15, 10),
    column="Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    legend=True,
    cmap="RdYlGn_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    #  scheme="natural_breaks"
)
ax.set_axis_off()

fig = ax.get_figure()
# fig.savefig("rentals_imp.png", dpi=300)

In [ ]:
ax = fi_mean.plot(
    figsize=(15, 10),
    column="Obyvatelstvo - s dlouhodobým pobytem - celkem",
    legend=True,
    cmap="RdYlGn_r",
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    #  scheme="natural_breaks"
)
ax.set_axis_off()

fig = ax.get_figure()
# fig.savefig("rentals_imp.png", dpi=300)

In [ ]:
fi_mean1 = fi_mean[["Počet obyvatel na dům","Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem","Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví"]]

In [ ]:
global_min=fi_mean1.min().min()

In [ ]:
global_max=fi_mean1.max().max()

In [ ]:
global_f1_max = max(
    row["model"].local_oob_f1_macro_.max() for _, row in df_models.iterrows()
)

global_f1_min = min(
    row["model"].local_oob_f1_macro_.min() for _, row in df_models.iterrows()
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 10))



cax = fig.add_axes((.94, 0.4, .02, .2))



fi_mean.plot(
    ax=axes[0],
    column="Počet obyvatel na dům",
    legend=False,
    cmap="magma",
    vmin=global_min,
    vmax=global_max,
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
)
axes[0].set_axis_off()
axes[0].set_title("Population Density", fontsize=10)


fi_mean.plot(
    ax=axes[1],
    column="Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    # legend=True,
    cmap="magma",
    # cax=cax,
    vmin=global_min,
    vmax=global_max,
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.8,"orientation": "horizontal"},
)

axes[1].set_axis_off()
axes[1].set_title("Res. in Owner-Occupied Dwellings",fontsize=10)

fi_mean.plot(
    ax=axes[2],
    column="Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    legend=True,
    cax=cax,
    cmap="magma",
    vmin=global_min,
    vmax=global_max,
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.02},
)
axes[0].add_artist(ScaleBar(1,location="lower left",height_fraction=0.015))


axes[2].set_axis_off()
axes[2].set_title("Empl. sector - Agriculture/Fishery",fontsize=10)
fig.subplots_adjust(wspace=0)
fig.align_titles()

#plt.tight_layout()
fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")
